# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: Triangulados

# Integrantes
* Posse Lucila
* Ventroni Milena
* Pili
* Fran


### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 8.


### Solución
#### 1.a. Rango y cantidad mínima de puntos

Para que el sistema homogéneo $Qm=0$ tenga una solución no nula única salvo un factor de escala, es necesario que el núcleo de $Q$ tenga dimensión 1. Por el teorema de rango-nulidad, dado que $Q$ tiene 12 columnas:

$$
\operatorname{rg}(Q)+\dim(\ker Q)=12
$$

Por lo tanto su rango debe ser exactamente 11.

Como cada correspondencia entre un punto 3D y su proyección 2D aporta dos filas a $Q$, se necesitan como mínimo 6 puntos:

$$
2n\geq11\implies n\geq6
$$

Sin embargo, no cualquier conjunto de 6 puntos es suficiente: deben encontrarse posicionados de una manera tal que la matriz $Q$ resultante tenga rango 11. En particular, <u>los puntos no pueden ser todos coplanares</u>.

**Justificación de la no coplanaridad**

Si todos los puntos 3D son coplanares, existe un vector no nulo $a\in\mathbb{R}^4$ tal que:

$$
a^TP_i=0 \qquad \forall i
$$

donde $P_i=(U_i,V_i,W_i,1)^T$.

Recordando que cada punto aporta dos filas a $Q$:

$$
Q_i=
\begin{pmatrix}
P_i^T & 0^T & -u_iP_i^T\\
0^T & P_i^T & -v_iP_i^T
\end{pmatrix}
$$

Como $a^TP_i=0$, los siguientes vectores pertenecen al núcleo de $Q$:

$$
m_1=
\begin{pmatrix}a\\0\\0\end{pmatrix},
\qquad
m_2=
\begin{pmatrix}0\\a\\0\end{pmatrix},
\qquad
m_3=
\begin{pmatrix}0\\0\\a\end{pmatrix}
$$

Estos tres vectores son linealmente independientes, ya que $a\neq0$ y ocupa bloques distintos en cada uno.

Además, el vector $m$ correspondiente a la matriz de proyección de la cámara también pertenece al núcleo de $Q$. Como dicha matriz representa una cámara válida, no puede tener todas sus filas proporcionales a $a^T$, por lo que $m$ es linealmente independiente de $m_1,m_2,m_3$.

Entonces:

$$
\dim(\ker Q)\geq4
$$

Por rango-nulidad:

$$
\operatorname{rg}(Q)=12-\dim(\ker Q)\leq8
$$

Por lo tanto, si todos los puntos son coplanares, $Q$ no puede alcanzar rango 11. Esto demuestra que los puntos no pueden ser todos coplanares para determinar una única matriz de proyección salvo escala.

#### 1.b. Minimización

Buscamos el vector $m$ que resuelve:

$$
\min_{\|m\|_2=1}m^TQ^TQm
=
\min_{\|m\|_2=1}\|Qm\|_2^2
$$

##### Descomposición en valores singulares (SVD)

Consideramos la descomposición en valores singulares de $Q$, continuando el item 1a. $Q\in\mathbb{R}^{12\times12}$:

$$
Q=U\Sigma V^T
$$

Entonces:

$$
\begin{aligned}
Q^TQ
&=(V\Sigma^TU^T)(U\Sigma V^T)\\
&=V\Sigma^2V^T
\end{aligned}
$$

donde utilizamos que $U^TU=I$ y que $\Sigma$ es diagonal, por lo que $\Sigma^T=\Sigma$.

Como $\Sigma^2$ es una matriz diagonal cuyos elementos son los cuadrados de los valores singulares, reemplazamos en la expresión a minimizar:

$$
\min_{\|m\|_2=1}m^TV\Sigma^2V^Tm
$$

Realizamos el cambio de variable:

$$
y^T=m^TV
\quad\Longrightarrow\quad
y=V^Tm
$$

Como $V$ es ortogonal:

$$
\|y\|_2^2=y^Ty=m^TVV^Tm=m^Tm=1
$$

De esta manera, el problema queda:

$$
\min_{\|y\|_2=1}y^T\Sigma^2y
$$

Considerando los valores singulares ordenados de mayor a menor:

$$
\sigma_1\geq\sigma_2\geq\cdots\geq\sigma_{12}\geq0
$$

Tenemos:

$$
\begin{aligned}
y^T\Sigma^2y
&=\sum_{i=1}^{12}\sigma_i^2y_i^2\\
&\geq\sigma_{12}^2\sum_{i=1}^{12}y_i^2\\
&=\sigma_{12}^2\|y\|_2^2\\
&=\sigma_{12}^2
\end{aligned}
$$

Por lo tanto:

$$
m^TQ^TQm\geq\sigma_{12}^2
$$

Para alcanzar esta cota inferior, tomamos:

$$
y=e_{12}=(0,\ldots,0,1)^T
$$

Este vector cumple $\|y\|_2=1$ y:

$$
y^T\Sigma^2y=\sigma_{12}^2
$$

Como $y=V^Tm$, recuperamos:

$$
m=Vy=Ve_{12}=v_{12}
$$

Es decir, $m$ es la última columna de $V$, asociada al menor valor singular de $Q$.

Por lo tanto:

$$
\boxed{
\min_{\|m\|_2=1}m^TQ^TQm=\sigma_{12}^2
}
$$

Y una solución es:

$$
\boxed{m=v_{12}}
$$

ya que los autovalores de $Q^TQ$ son los cuadrados de los valores singulares de $Q$:

$$
\boxed{
\lambda_{\min}(Q^TQ)=\sigma_{12}^2
}
$$

Por lo tanto, podemos obtener el vector $m$ como un autovector unitario asociado al menor autovalor de $Q^TQ$, o equivalentemente como la última columna de $V$ en la descomposición SVD de $Q$.


### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



### Solución
#### Ejercicio 2.a — Construcción de la matriz Q

A partir de las correspondencias entre los puntos del mundo real y sus proyecciones en la imagen, construimos la matriz $Q$ mediante la función `obtenerQ3d(imgPts, objPts)`, siguiendo la ecuación (6) del enunciado.

Los datos contienen 12 puntos tridimensionales y 210 capturas con sus respectivas coordenadas de imagen.

Para la primera captura, cada correspondencia aporta dos filas:

$$
Q_i=
\begin{pmatrix}
P_i^T & 0^T & -u_iP_i^T\\
0^T & P_i^T & -v_iP_i^T
\end{pmatrix}
$$

donde $P_i=(U_i,V_i,W_i,1)^T$.

Al tener en cuenta las 12 correspondencias, obtenemos:

$$
\boxed{Q\in\mathbb{R}^{24\times12}}
$$



In [38]:
import numpy as np
import pandas as pd
import os
import alc


# Ejercicio 2.a - Construimos Q

ruta_puntos_3d = os.path.join("calib", "3d", "p_W_corners.txt")
ruta_puntos_2d = os.path.join("calib", "3d", "detected_corners.txt")

objPts = np.loadtxt(ruta_puntos_3d, delimiter=",")
datos = np.loadtxt(ruta_puntos_2d)
imgPts = datos[0].reshape(12, 2)

Q_3d = alc.obtenerQ3d(imgPts, objPts)
print("Dimensiones de Q", Q_3d.shape)

# Mostramos una versión resumida de Q
np.set_printoptions(threshold=50, precision=2, suppress=True)

print("\nMatriz Q (resumida):")
print(Q_3d)


Dimensiones de Q (24, 12)

Matriz Q (resumida):
[[     0.        8.        0.   ...  -1492.       -0.     -186.5 ]
 [     0.        0.        0.   ...  -1500.       -0.     -187.5 ]
 [    10.4       8.        0.   ...  -2116.       -0.     -264.5 ]
 ...
 [     0.        0.        0.   ...  -7012.5    3071.47   -280.5 ]
 [    19.4      25.      -24.45 ... -12412.5   12139.42   -496.5 ]
 [     0.        0.        0.   ...  -7512.5    7347.22   -300.5 ]]



### Ejercicio 2.b — Matriz de proyección

A partir de la matriz $Q$ obtenida en el inciso anterior, calculamos la matriz $Q^TQ$ y aplicamos la función `diagRH` del módulo `alc` para obtener sus autovalores y autovectores.

Como vimos en el ejercicio 1.b, el vector unitario que minimiza $\|Qm\|_2^2$ es un autovector asociado al menor autovalor de $Q^TQ$.

La matriz obtenida cumple la restricción $\|m\|_2=1$ y presenta un residuo:

$$
\|Qm\|_2\approx0.00822
$$

Además, el cuadrado de dicho residuo coincide aproximadamente con el menor autovalor de $Q^TQ$, como se esperaba por el resultado del ejercicio 1.b.


In [39]:
# Ejercicio 2.b - Recuperar la matriz de proyección

M = alc.resolverDLT3D(Q_3d)

print("Dimensiones de M:", M.shape)
print("\nMatriz de proyección M:")
print(M)

# Comprobamos la norma del vector m
m = M.reshape(12)

print("\nNorma de m:", alc.norma(m, 2))

# Calculamos el residuo del sistema
residuo = alc.matvec(Q_3d, m)

print("Norma de Qm:", alc.norma(residuo, 2))


Dimensiones de M: (3, 4)

Matriz de proyección M:
[[-0.06 -0.02  0.01 -0.84]
 [-0.   -0.06 -0.01 -0.54]
 [-0.   -0.   -0.   -0.01]]

Norma de m: 1.0
Norma de Qm: 0.008222745610322008


In [ ]:
# ESTE EJEMPLO FALLA CON LA IMPLEMENTACIÓN ACTUAL 
#fijense que el determinante da -1. No es una matriz de rotación valida. 
#Creo que es un error al normalizar 

escala = alc.norma(M[2, :3], 2)

M_normalizada = M / escala

print("Matriz normalizada:")
print(M_normalizada)

print("\nNorma de la tercera fila:")
print(alc.norma(M_normalizada[2, :3], 2))

R, t, K = alc.devolverMatricesCalibracion3d(M_normalizada)

print("Matriz K:")
print(K)

print("\nMatriz R:")
print(R)

print("\nVector t:")
print(t)

RtR = alc.multiplicacion_matrices(alc.transpuesta(R), R)

print("R^T R:")
print(RtR)

print("\nDiferencia respecto de la identidad:")
print(RtR - np.eye(3))

det_R = (
    R[0, 0] * (R[1, 1] * R[2, 2] - R[1, 2] * R[2, 1])
    - R[0, 1] * (R[1, 0] * R[2, 2] - R[1, 2] * R[2, 0])
    + R[0, 2] * (R[1, 0] * R[2, 1] - R[1, 1] * R[2, 0])
)

print("Determinante de R:", det_R)

Matriz normalizada:
[[ -511.81  -180.81   111.04 -6649.53]
 [  -29.36  -484.28  -104.12 -4268.42]
 [   -0.7     -0.35    -0.62   -40.59]]

Norma de la tercera fila:
1.0
Matriz K:
[[425.86  -0.02 354.42]
 [  0.   424.57 256.85]
 [  0.     0.     1.  ]]

Matriz R:
[[-0.62 -0.13  0.78]
 [ 0.36 -0.93  0.13]
 [-0.7  -0.35 -0.62]]

Vector t:
[ 18.17  14.5  -40.59]
R^T R:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Diferencia respecto de la identidad:
[[-0.  0.  0.]
 [ 0. -0.  0.]
 [ 0.  0.  0.]]
Determinante de R: -1.0


### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.


### Solución
#### Ejercicio 3.a — Construcción de la matriz Q

Para obtener la matriz de parámetros intrínsecos $K$, utilizamos el método de calibración con puntos coplanares.

El tablero contiene 77 esquinas internas, distribuidas en 11 columnas y 7 filas, separadas por 30 mm. Como todos los puntos pertenecen a un mismo plano, consideramos sus coordenadas $(U,V)$, tomando $W=0$.

Para cada imagen, construimos una matriz $Q_j$ a partir de las correspondencias entre los puntos del tablero y sus proyecciones. Cada correspondencia aporta dos ecuaciones, por lo que:

$$
Q_j\in\mathbb{R}^{154\times9}.
$$

Luego, mediante la función `resolverDLT2D`, obtenemos la homografía $H_j$ asociada a cada vista, que relaciona las coordenadas del tablero con las coordenadas de la imagen:

$$
\lambda
\begin{pmatrix}
u\\v\\1
\end{pmatrix}
=
H_j
\begin{pmatrix}
U\\V\\1
\end{pmatrix}.
$$

A partir de las homografías, construimos la matriz $A$ mediante la función `armarA`. Finalmente, utilizamos `calcularMatrizK2D` para recuperar la matriz de parámetros intrínsecos:

$$
K=
\begin{pmatrix}
\alpha_u & s & u_0\\
0 & \alpha_v & v_0\\
0 & 0 & 1
\end{pmatrix}.
$$

Para este cálculo utilizamos las diez imágenes disponibles en la carpeta de calibración.


In [34]:
# Cargamos las coordenadas de las esquinas
ruta_coplanar = os.path.join("calib", "coplanar", "marcadas")
ruta_esquinas = os.path.join(ruta_coplanar, "esquinas.csv")

esquinas = pd.read_csv(ruta_esquinas)

# Construimos Q y calculamos H para cada imagen
matrices_Q = {}
matrices_H = {}
Hlist = []

for imagen in os.listdir(ruta_coplanar):

    if imagen != "esquinas.csv": #no me gusta esto, ver cómo cambiar (ESTA EN NOTAS)

        # Seleccionamos las correspondencias de la imagen
        datos_img = esquinas[esquinas["imagen"] == imagen]

        objPts = np.asarray(datos_img[["U", "V"]])
        imgPts = np.asarray(datos_img[["u", "v"]])

        # Construimos Q
        Q = alc.obtenerQ2d(imgPts, objPts)
        matrices_Q[imagen] = Q

        # Recuperamos la homografía
        H = alc.resolverDLT2D(Q)
        matrices_H[imagen] = H
        Hlist.append(H)

        print(f"{imagen}: Q {Q.shape}, H {H.shape}")

# Construimos la matriz A
A = alc.armarA(Hlist)

print("\nCantidad de homografías:", len(Hlist))
print("Dimensiones de A:", A.shape)

# Calculamos la matriz K
K = alc.calcularMatrizK2D(A)

print("\nMatriz de parámetros intrínsecos K:")
print(K)


Im_R_1.png: Q (154, 9), H (3, 3)
Im_R_10.png: Q (154, 9), H (3, 3)
Im_R_2.png: Q (154, 9), H (3, 3)
Im_R_3.png: Q (154, 9), H (3, 3)
Im_R_4.png: Q (154, 9), H (3, 3)
Im_R_5.png: Q (154, 9), H (3, 3)
Im_R_6.png: Q (154, 9), H (3, 3)
Im_R_7.png: Q (154, 9), H (3, 3)
Im_R_8.png: Q (154, 9), H (3, 3)
Im_R_9.png: Q (154, 9), H (3, 3)

Cantidad de homografías: 10
Dimensiones de A: (20, 6)

Matriz de parámetros intrínsecos K:
[[739.97  -0.75 520.35]
 [  0.   751.25 299.25]
 [  0.     0.     1.  ]]



#### Ejercicio 3.b — Recuperación de la pose

Una vez obtenida la matriz de parámetros intrínsecos $K$, recuperamos la pose de la cámara para las vistas solicitadas.

Para un tablero coplanar, la homografía satisface:

$$
H_j \sim K
\begin{pmatrix}
r_1&r_2&t
\end{pmatrix},
$$

donde $r_1$ y $r_2$ son las primeras dos columnas de la matriz de rotación $R$, y $t$ es el vector de traslación.

Utilizamos la función `obtenerPose2D(Hj, K)` del módulo `alc` para recuperar la matriz de rotación $R_j$ y el vector de traslación $t_j$ de cada vista.

Para verificar las rotaciones obtenidas, comprobamos que satisfagan aproximadamente $R_j^TR_j=I$ y que su determinante sea cercano a $1$.


In [ ]:
# Ejercicio 3.b - Recuperar la pose de las vistas solicitadas

poses = {}
imagenes_solicitadas = ["Im_R_1.png", "Im_R_10.png", "Im_R_20.png"]

if K is None:
    print("No se pudo calcular K. Revisar el ejercicio 3.a.")

else:
    for imagen in imagenes_solicitadas:

        if imagen not in matrices_H:
            print(f"\n{imagen}: no hay correspondencias disponibles.") #Salva lo de la imágen 20, me agarró amnesia no me acuerdo si dijeron algo de eso.
            continue

        # Recuperamos la pose
        H = matrices_H[imagen]
        R, t = alc.obtenerPose2D(H, K)

        poses[imagen] = (R, t)

        print(f"\n--- {imagen} ---")

        print("Matriz R:")
        print(R)

        print("\nVector t:")
        print(t)

        # Verificamos ortogonalidad
        RtR = alc.multiplicacion_matrices(
            alc.transpuesta(R), R
        )

        print("\nR^T R:")
        print(RtR)

        # Determinante de R (fórmula para matrices 3x3)
        det_R = (
            R[0, 0] * (R[1, 1]*R[2, 2] - R[1, 2]*R[2, 1])
            - R[0, 1] * (R[1, 0]*R[2, 2] - R[1, 2]*R[2, 0])
            + R[0, 2] * (R[1, 0]*R[2, 1] - R[1, 1]*R[2, 0])
        )

        print("\nDeterminante de R:", det_R)



--- Im_R_1.png ---
Matriz R:
[[ 1.   -0.01 -0.03]
 [ 0.01  1.    0.01]
 [ 0.03 -0.01  1.  ]]

Vector t:
[-109.2  -141.83  546.51]

R^T R:
[[ 1. -0.  0.]
 [-0.  1.  0.]
 [ 0.  0.  1.]]

Determinante de R: 1.0000000000000002

--- Im_R_10.png ---
Matriz R:
[[ 1.    0.07 -0.02]
 [-0.06  0.99  0.16]
 [ 0.03 -0.16  0.99]]

Vector t:
[ -96.4  -154.47  537.37]

R^T R:
[[ 1. -0. -0.]
 [-0.  1.  0.]
 [-0.  0.  1.]]

Determinante de R: 1.0

Im_R_20.png: no hay correspondencias disponibles.


### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---